In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

import gdp_analysis
from gdp_analysis.load import load_gdp
from gdp_analysis.clean import clean_gdp, summarise_gdp
from gdp_analysis.plot import plot_gdp

ROOT = Path(gdp_analysis.__file__).resolve().parents[1]
print("Project folder:", ROOT)

In [ ]:
raw = load_gdp(ROOT / "data" / "raw" / "gdp_raw.json")
data = clean_gdp(raw)

expected_countries = {
    "GBR", "USA", "BRA", "JPN", "CHN", "DEU", "CHE"
}

assert set(data["country_code"]) == expected_countries
assert len(data) == 7 * 23
assert data["gdp"].gt(0).all()

display(
    data.groupby("country")["year"].agg(["min", "max", "count"])
)

output_folder = ROOT / "data" / "processed"
output_folder.mkdir(parents=True, exist_ok=True)

data.to_csv(output_folder / "gdp_clean.csv", index=False)

In [ ]:
figure_folder = ROOT / "figures"
figure_folder.mkdir(parents=True, exist_ok=True)

fig, ax = plot_gdp(data)
fig.savefig(
    figure_folder / "gdp_levels.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

fig, ax = plot_gdp(data, indexed=True)
fig.savefig(
    figure_folder / "gdp_index.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

In [ ]:
summary = summarise_gdp(data)
display(summary.round(2))

for country, row in summary.iterrows():
    print(
        f"{country}: real GDP changed by "
        f"{row['change_percent']:.1f}% between 2000 and 2022."
    )